# Sesión 12 — Análisis de accesibilidad (IHC · UNMSM)

**Entradas:** `S12_eventos_accesibilidad.csv` y `S12_hallazgos_accesibilidad.csv` exportados por la aplicación HTML. **No hay datos prefabricados.**

Este cuaderno calcula únicamente estadísticas descriptivas de eventos observados y una matriz de prioridades de hallazgos documentados por un evaluador. **No certifica conformidad WCAG** ni sustituye evaluación manual, uso de tecnologías de apoyo ni participación de personas con discapacidad.

**Flujo:** instalaciones → carga de archivos → control de calidad → métricas → gráficos → priorización → CSV de resumen.


In [ ]:
# ES: Instalar explícitamente dependencias para Colab gratuito. EN: Install dependencies in free Colab.
%pip -q install pandas matplotlib

In [ ]:
# ES: Bibliotecas de análisis; no se generan datos sintéticos. EN: Analysis libraries; no synthetic data are created.
from pathlib import Path
from io import BytesIO
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# ES: Contratos de los dos archivos. EN: Required columns for both exported files.
EVENT_COLS = ["timestamp", "mode", "event_type", "detail", "elapsed_ms", "clicks", "errors", "keyboard_actions"]
FIND_COLS = ["timestamp", "mode", "criterion", "severity", "evidence", "recommendation"]
print("Entorno preparado; cargue los CSV de la interfaz.")

In [ ]:
# ES: Abrir selector de archivos de Colab; en Jupyter local admite ficheros en el directorio de trabajo. EN: Colab upload with local Jupyter fallback.
try:
    from google.colab import files
    uploaded = files.upload()
    sources = {name: BytesIO(data) for name, data in uploaded.items()}
except ImportError:
    sources = {p.name: p for p in Path('.').glob('S12_*accesibilidad.csv')}
    print("Jupyter local: fuentes localizadas:", list(sources))

# ES: La función valida datos reales sin inventar filas. EN: Validate uploaded, real observations without fabricating records.
def load_csv(filename, required):
    """Lee CSV UTF-8 con BOM y comprueba el contrato mínimo de columnas."""
    if filename not in sources:
        print(f"No se proporcionó {filename}. Se usará una tabla vacía, sin resultados inventados.")
        return pd.DataFrame(columns=required)
    source = sources[filename]
    if hasattr(source, 'seek'):
        source.seek(0)
    df = pd.read_csv(source, encoding='utf-8-sig')
    missing = sorted(set(required)-set(df.columns))
    if missing:
        raise ValueError(f"En {filename} faltan columnas: {missing}")
    return df[required].copy()

events = load_csv('S12_eventos_accesibilidad.csv', EVENT_COLS)
findings = load_csv('S12_hallazgos_accesibilidad.csv', FIND_COLS)
print(f"Eventos: {len(events)} | Hallazgos humanos: {len(findings)}")

In [ ]:
# ES: Conversión segura de tipos y limpieza sin alterar el origen. EN: Safely parse types and flag missing values.
if not events.empty:
    events['timestamp'] = pd.to_datetime(events['timestamp'], errors='coerce', utc=True)
    for col in ['elapsed_ms','clicks','errors','keyboard_actions']:
        events[col] = pd.to_numeric(events[col], errors='coerce')
    events = events.dropna(subset=['timestamp', 'event_type', 'mode'])
if not findings.empty:
    findings['timestamp'] = pd.to_datetime(findings['timestamp'], errors='coerce', utc=True)
    findings['severity'] = pd.to_numeric(findings['severity'], errors='coerce')
    bad = findings['severity'].notna() & ~findings['severity'].between(0,4)
    if bad.any():
        raise ValueError('El campo severity debe ser 0, 1, 2, 3 o 4.')
    findings = findings.dropna(subset=['timestamp', 'criterion', 'evidence'])
print('Filas válidas:', len(events), len(findings))

In [ ]:
# ES: Resumir tipos de eventos observados por modo de interfaz. EN: Summarize measured events by interface mode.
if events.empty:
    print('Todavía no hay eventos. Complete una tarea consentida en el HTML y exporte el CSV.')
else:
    event_summary = events.groupby(['mode','event_type'], dropna=False).size().rename('eventos').reset_index()
    display(event_summary)
    fig, ax = plt.subplots(figsize=(9,4))
    event_summary.pivot(index='event_type', columns='mode', values='eventos').fillna(0).plot(kind='bar', ax=ax)
    ax.set(title='Eventos reales por modo', xlabel='Tipo de evento', ylabel='Frecuencia observada')
    plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
# ES: Solo se calculan duraciones de eventos explícitamente completados. EN: Calculate task durations only when a completion event exists.
complete = events.loc[events['event_type']=='reserva_completada'].copy() if not events.empty else pd.DataFrame(columns=EVENT_COLS)
if complete.empty:
    print('No hay tareas completas: la tasa de finalización no se infiere de eventos parciales.')
else:
    complete['duration_s'] = complete['elapsed_ms']/1000
    complete = complete.loc[complete['duration_s'].ge(0)]
    task_metrics = complete.groupby('mode')['duration_s'].agg(n='count', media_s='mean', mediana_s='median', minimo_s='min', maximo_s='max').round(2)
    display(task_metrics)
    print('Interpretación: son observaciones del navegador, NO estadísticas poblacionales.')

## Tasa de éxito y limitaciones

En este instrumento, cada evento `inicio_tarea` marca un intento observado y `reserva_completada` una finalización. Si faltan sesiones o se cierra el navegador sin finalizar, el cociente **no** representa por sí solo la tasa de éxito del estudio. Cada estudiante debe registrar las condiciones y analizar el sesgo de sesiones incompletas.

**Formato Markdown:** `TE_observada = finalizaciones / inicios × 100`.

**Formato LaTeX:** $TE_{obs}=\frac{N_{fin}}{N_{ini}}\times 100$.

**Formato texto:** tasa observada igual al número de eventos de finalización dividido por el número de eventos de inicio, multiplicado por cien.

**Lectura verbal:** *Se divide el número de tareas completadas entre los inicios registrados, recordando que un mismo navegador puede contener múltiples intentos.*


In [ ]:
# ES: Esta medida se informa como cociente descriptivo y no como tasa poblacional. EN: Descriptive attempt ratio, not a population-level success rate.
starts = int(events['event_type'].eq('inicio_tarea').sum()) if not events.empty else 0
ends = int(events['event_type'].eq('reserva_completada').sum()) if not events.empty else 0
if starts:
    print(f'Inicio(s): {starts} | Finalización(es): {ends} | Cociente descriptivo: {100*ends/starts:.1f}%')
    if ends>starts:
        print('ADVERTENCIA: más finalizaciones que inicios; compruebe el protocolo.')
else:
    print('No hay inicios observados; no es posible calcular cociente de finalización.')

In [ ]:
# ES: Priorizar barreras reales, sin generar evaluaciones de ejemplo. EN: Rank only findings entered by a human evaluator.
if findings.empty:
    print('Aún no existen hallazgos humanos documentados.')
else:
    table = findings[['mode','criterion','severity','evidence','recommendation']].sort_values('severity', ascending=False)
    display(table)
    counts = findings.groupby(['criterion','severity']).size().rename('frecuencia').reset_index()
    display(counts)
    fig, ax = plt.subplots(figsize=(10,4))
    findings.groupby('criterion').size().sort_values().plot(kind='barh', ax=ax)
    ax.set(title='Frecuencia de hallazgos documentados por criterio', xlabel='Hallazgos observados', ylabel='Criterio WCAG')
    plt.tight_layout();plt.show()
    print('La severidad 0–4 es una escala DIDÁCTICA de priorización, no un nivel de conformidad WCAG.')

In [ ]:
# ES: Preparar reporte exportable, dejando vacíos los cálculos sin datos. EN: Export summary without invented metrics.
report = pd.DataFrame([{
    'fecha_analisis_utc': pd.Timestamp.now(tz='UTC').isoformat(),
    'n_eventos': len(events),
    'n_hallazgos': len(findings),
    'n_inicios': starts,
    'n_finalizaciones': ends,
    'cociente_finalizacion_pct': round(ends/starts*100,2) if starts else None,
    'n_hallazgos_impacto_3_4': int(findings['severity'].ge(3).sum()) if not findings.empty else 0,
    'alcance': 'Descriptivo; no es certificacion WCAG'
}])
report.to_csv('S12_resumen_accesibilidad.csv', index=False, encoding='utf-8-sig')
display(report)
try:
    from google.colab import files
    files.download('S12_resumen_accesibilidad.csv')
except ImportError:
    print('Archivo disponible en el directorio local: S12_resumen_accesibilidad.csv')

## Discusión, integridad académica y referencias

- ¿Qué diferencias descubre el teclado que no se evidencian con una comprobación automática?
- ¿Por qué una pantalla corregida no puede declararse conforme solo por superar cuatro verificaciones de DOM?
- ¿Qué barreras inciden en acceso, comprensión, movilidad, audición, visión y necesidades cognitivas?
- Registre evidencia observada; no atribuya opiniones o resultados a usuarios inexistentes.

Fuentes oficiales:

- https://www.w3.org/WAI/WCAG22/quickref/
- https://www.w3.org/WAI/standards-guidelines/wcag/new-in-22/
- https://www.w3.org/TR/wcag-em-2/

**Uso de IA:** GPT-6 (OpenAI), 20 de octubre de 2026, asistencia en elaboración de código, documentación y revisión. Todas las conclusiones de accesibilidad quedan sujetas a validación humana.
